# Study 09 · 00 · Data leakage audit

Before any experiment, this notebook checks that nothing a forecast uses at its origin depends on information from after that origin. It has two parts:

1. **Code review (static):** where each input enters the pipeline and what guards it.
2. **Dynamic tests:** replace the future with random values and verify that what the model sees at the origin does not change. A positive control (the historical lag-0 weather alignment) shows the test does detect look-ahead when it exists.

Shared code: `research/09_epidemics_revision/code/` (`common.py`, `leakage.py`). Inputs are checked against the SHA-256 hashes recorded in `research/08_v3_confirmed_cases/manifiesto_b3_v4.json`. The protocol is `research/09_epidemics_revision/protocolo.md`.

## 1 · Code review

| Area | Where | Finding | Verdict |
|---|---|---|---|
| Case-history predictors (lags, 4-week mean, positive weeks) | `features/history_calendar.py` | shifted by the horizon: only weeks ≤ origin | no leakage (tested below) |
| Onset predictors (time since last case, slope, previous season) | `features/onset_features.py` | each origin week uses weeks ≤ that week; the previous season is complete | no leakage (tested below) |
| Calendar and geography | `history_calendar.py`, geography file | week o+2 is known in advance; geography is static | OK |
| Outbreak threshold | `features/outbreak_label.future_outbreak_threshold` | same epi-week of the 5 previous calendar years, ≥ 48 weeks before the origin | no leakage (tested below) |
| Weather, primary arm (lag 2) | `run_spatial_controls.weekly_fields` | 5-week means ending 2 weeks before the origin week; error if any value was published after the origin close (ERA5-Land = week end + 7 d; CHIRPS = pentad end + 5 d) | no leakage (tested below) |
| Weather, historical alignment (lag 0) | same, `lag=0` | includes the origin week and accepts publication up to origin + 14 d | **look-ahead by design**; kept only as a contrast (E2) |
| Niño 1+2 | `phase4_enso.enso_at_origins`, guard G21 | latest month with availability date ≤ origin close | no leakage (tested below) |
| Training set per block | `model/train._split_block`, `probabilistic_output.fit_e3` | targets closed by the block's first test origin; error on overlap | OK |
| Platt, count blend, NB2 dispersion | `protocol.calibrate_chronologically`, `phase5_counts.mix_chronologically`, `fit_e3` | earlier blocks, labels closed by the cut-off; α fitted in-sample on training | OK (refit test in notebook 01) |
| Probabilistic persistence, climatology | `run_statistical_benchmarks.reference_scores`, `pdw.climatology` | chronological calibration; labels closed by the cut-off | OK |
| Alert cut-offs | `evaluate_alert_cutoffs.py` | chosen on 2009, 2012–2014, 2018, 2021; applied to 2022, 2023, 2026; order is checked | OK |
| Tuned (equal-budget) arm | `run_symmetric_tuning.py` | tuning seasons 2018 and 2021 come after evaluation seasons 2010, 2015, 2016 | **temporal leakage in the tuned arm**: not used in the main analyses |
| Fixed hyperparameters and base predictors | `frozen_configs.CONFIG_2026_V1` | chosen in studies 04–05 on blocks 2021–2025, which overlap the evaluation | **selection shared by both arms**: can inflate absolute skill, much less the paired climate contrast |
| Choice of b3 among 12 climate representations | study 06, climate matrix | selected on the six decision seasons | **selection favouring climate** on those seasons; conservative for a null result |
| Finalized surveillance counts | silver/confirmed | final counts, not the values published at the time | **not available in real time**; cannot be removed without data vintages |
| Weather product timing | CHIRPS v3 Final on the Prelim schedule; today's ERA5-Land latency applied to past seasons | values at least as good as real time | **mild optimism for climate** |

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

import leakage as lk

c.environment()

{'python': '3.12.0',
 'plataforma': 'macOS-26.5.1-arm64-arm-64bit',
 'xgboost': '3.4.1',
 'pandas': '3.0.6',
 'numpy': '2.5.3',
 'sklearn': '1.9.1',
 'xgb_settings': {'device': 'cpu', 'n_jobs': 2},
 'variables': {'DENGUEKAY_XGB_DEVICE': 'cpu',
  'DENGUEKAY_XGB_NJOBS': '2',
  'OMP_NUM_THREADS': '2',
  'OPENBLAS_NUM_THREADS': '1',
  'VECLIB_MAXIMUM_THREADS': '1'}}

## 2 · Inputs equal to the paper's (SHA-256 against the V4 manifest)

In [2]:
c.check_sources()

,fuente,archivo,sha256_manifiesto,sha256_local,coincide
0,data/silver/confirmados/meteo_socio_epi_piura_2017_2025.csv,data/silver/confirmados/meteo_socio_epi_piura_2017_2025.csv,ad0a9917d154,ad0a9917d154,True
1,data/silver/confirmados/epi_piura_semanal.csv,data/silver/confirmados/epi_piura_semanal.csv,6b5b62e8fbc4,6b5b62e8fbc4,True
2,data/silver/confirmados/epi_sala_semanal.csv,data/silver/confirmados/epi_sala_semanal.csv,ff3354ca19cf,ff3354ca19cf,True
3,data/silver/confirmados/meteo_socio_epi_piura_2017_2025....,data/silver/confirmados/meteo_socio_epi_piura_2017_2025....,865f1338d575,865f1338d575,True
4,data/silver/meteo_humedad_era5_land_2000_2026.csv,data/silver/meteo_humedad_era5_land_2000_2026.csv,3e91e57f31fa,3e91e57f31fa,True
5,data/gold/manifest_fase6.json,data/gold/manifest_fase6.json,024c654f9b5a,024c654f9b5a,True
6,data/reference/catalogo_ubigeos_piura.csv,data/reference/catalogo_ubigeos_piura.csv,5ca987e295ba,5ca987e295ba,True
7,precip_chirps_v3_semanal_desde_2000.csv,data/silver/precip_chirps_v3_semanal_desde_2000.csv,e4cd57a5927c,e4cd57a5927c,True
8,enso_nino12_mensual.csv,data/silver/enso_nino12_mensual.csv,3250fdfa62fb,3250fdfa62fb,True
9,geografia,models/experimentos/revision_epidemics/insumos/geografia...,b56bacee763b,b56bacee763b,True


## 3 · Case history: replace every count after the origin with random values

Eight origins between 2008 and 2026, including the 2017 and 2023 epidemics and the switch from the national file to the Sala dashboard. For each one, all counts after the origin week (all 65 districts) are replaced with random integers 0–399 and the long inputs are rebuilt. The predictors at the origin and the thresholds and persistence of its targets t+2…t+4 must be identical. As a control, the share of **later** origins whose predictors changed shows that the perturbation is real.

In [3]:
inputs, quality = c.load_inputs(validate=True)
print("Great Expectations on the confirmed panel:", {k: quality[k] for k in ("exito", "evaluadas", "fallidas") if k in quality} or quality)
base_columns, hp = c.archive.original.p3._config()
closes = sorted(inputs["origenes"].origen_cierre.unique())
targets = ["2008-09-06", "2012-01-07", "2015-03-07", "2017-03-04", "2020-06-06", "2023-03-04", "2025-12-27", "2026-09-26"]
chosen = [max(x for x in closes if x <= pd.Timestamp(t)) for t in targets]
case_result = lk.case_history_test(inputs, chosen, base_columns)
case_result

Great Expectations on the confirmed panel: {'exito': True}


,origen_cierre,filas_futuras_alteradas,objetivos_comparados,predictores_iguales,umbrales_y_persistencia_iguales,control_origenes_posteriores_cambian
0,2008-09-06,61425,195,True,True,1.0
1,2012-01-07,50115,195,True,True,1.0
2,2015-03-07,39390,195,True,True,1.0
3,2017-03-04,32630,195,True,True,1.0
4,2020-06-06,21580,195,True,True,1.0
5,2023-03-04,12285,195,True,True,1.0
6,2025-12-27,2730,195,True,True,1.0
7,2026-09-26,195,130,True,True,NaN


## 4 · Weather and Niño 1+2: replace every value published after the origin close

Three alignments are rebuilt from the perturbed sources:
- **lag-2:** the paper's primary arm `b3_obs`.
- **lag-1:** ERA5-Land one week before the origin, CHIRPS two weeks before (protocol D3).
- **lag-0 historical:** the alignment of the original development, kept as the positive control.

The table counts districts whose weather predictors (b3 and d3 columns) changed at that origin. First, the study's per-source function is checked to be identical to `run_spatial_controls.weekly_fields` at lags 0 and 2.

In [4]:
weather_result, equivalence = lk.weather_test(chosen)
print(equivalence)
weather_result

{'lag 0': 'identical to run_spatial_controls.weekly_fields', 'lag 2': 'identical to run_spatial_controls.weekly_fields'}


,origen_cierre,lag0_historico: distritos con valores alterados,lag1: distritos con valores alterados,lag2_obs: distritos con valores alterados
0,2008-09-06,65,0,0
1,2012-01-07,65,0,0
2,2015-03-07,65,0,0
3,2017-03-04,65,0,0
4,2020-06-06,65,0,0
5,2023-03-04,65,0,0
6,2025-12-27,65,0,0
7,2026-09-26,65,0,0


In [5]:
assert case_result["predictores_iguales"].all() and case_result["umbrales_y_persistencia_iguales"].all()
assert (case_result["control_origenes_posteriores_cambian"].dropna() > 0).all()
lag2 = weather_result.filter(like="lag2").to_numpy()
lag1 = weather_result.filter(like="lag1").to_numpy()
lag0 = weather_result.filter(like="lag0").to_numpy()
assert (lag2 == 0).all() and (lag1 == 0).all(), "issue-time weather arms see future information"
assert (lag0 > 0).all(), "positive control failed: the test would not detect look-ahead"
print("PASSED: no future information reaches the case predictors, thresholds or the lag-1/lag-2 weather; "
      "the historical lag-0 alignment does use weather published after the origin.")

PASSED: no future information reaches the case predictors, thresholds or the lag-1/lag-2 weather; the historical lag-0 alignment does use weather published after the origin.


## 5 · Reading

- The predictors of the paper's primary comparison (`historia_geo` vs `b3_obs`) and the new lag-1 arm use only information available at the origin close.
- The historical alignment that produced the earlier −1.14% "gain" uses weather published after the origin: the positive control confirms the look-ahead (E2 tests how much it mattered).
- These tests can't rule out the risks marked in bold in §1: model and feature **selection** on overlapping seasons, **finalized** surveillance counts, and better-than-real-time weather values. They are reported as limitations. The tuned arm is not used in the main analyses because of its temporal leakage.
- The block-level check of training and calibration windows runs in notebook 01, on the fitted arms.